# Use case 1 — USDM v4 as a queryable data dictionary

**Question.** Treating `usdm_v4.ttl` as a queryable data dictionary, what
do I get?

**Pinned to v0.4.0.** This notebook reads the binding columns
(`usdm:boundCodelist` / `usdm:boundCodelistNote`) added in v0.1. Since
v0.4.0 every NCIt reference carries two IRI forms (EVS identifier and
OBO PURL — decision D4), so anchor and binding columns come in paired
`_evs` / `_obo` form. On a pre-v0.4.0 graph the `_obo` columns are empty.

**As-declared view.** Each class shows only its own attributes, not
inherited ones. Inheritance walking is case 3.


In [1]:
import rdflib
import pandas as pd
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
EXPECTED_BASELINE = 8642

if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/10/20 first to regenerate."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
n_triples = len(g)
print(f"parsed {n_triples} triples from {TTL_PATH}")
if n_triples != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.4.0 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.4.0 anchor shape")

def short(uri):
    """Last path segment, fragment-aware (handles NCIt #Cxxx, XSD #string, USDM /Class)."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]


parsed 8642 triples from ../usdm_v4.ttl


## Step 1 — class catalog

Every `owl:Class` IRI with its NCIt anchor, modifier, and superclass
count. One row per class.


In [2]:
CLASS_SPARQL = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?cls ?label ?ncit_evs ?ncit_obo ?prefLabel ?definition ?modifier (COUNT(?super) AS ?n_super)
WHERE {
  ?cls a owl:Class ; rdfs:label ?label .
  FILTER(isIRI(?cls))
  OPTIONAL {
    ?cls skos:exactMatch ?ncit_evs .
    FILTER(STRSTARTS(STR(?ncit_evs), "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"))
  }
  OPTIONAL {
    ?cls skos:exactMatch ?ncit_obo .
    FILTER(STRSTARTS(STR(?ncit_obo), "http://purl.obolibrary.org/obo/NCIT_"))
  }
  OPTIONAL { ?cls skos:prefLabel ?prefLabel }
  OPTIONAL { ?cls skos:definition ?definition }
  OPTIONAL { ?cls usdm:modifier ?modifier }
  OPTIONAL { ?cls rdfs:subClassOf ?super . FILTER(isIRI(?super)) }
}
GROUP BY ?cls ?label ?ncit_evs ?ncit_obo ?prefLabel ?definition ?modifier
ORDER BY ?label
"""

rows = []
for r in g.query(CLASS_SPARQL):
    rows.append({
        "class":      short(r["cls"]),
        "label":      str(r["label"]),
        "ncit_evs":   short(r["ncit_evs"]),
        "ncit_obo":   short(r["ncit_obo"]),
        "prefLabel":  str(r["prefLabel"]) if r["prefLabel"] else None,
        "definition": str(r["definition"]) if r["definition"] else None,
        "modifier":   str(r["modifier"]) if r["modifier"] else None,
        "n_super":    int(r["n_super"]),
    })
classes_df = pd.DataFrame(rows)
print(f"{len(classes_df)} classes")
classes_df.head(10)


86 classes


,class,label,ncit_evs,ncit_obo,prefLabel,definition,modifier,n_super
0,Abbreviation,Abbreviation,C42610,NCIT_C42610,Abbreviation,A set of letters that are drawn from a word or...,Concrete,0
1,Activity,Activity,C71473,NCIT_C71473,Study Activity,"An action, undertaking, or event, which is ant...",Concrete,0
2,Address,Address,C25407,NCIT_C25407,Address,A standardized representation of the location ...,Concrete,0
3,AdministrableProduct,AdministrableProduct,C215492,NCIT_C215492,Administrable Product,Any study product that is formulated and prese...,Concrete,0
4,AdministrableProductIdentifier,AdministrableProductIdentifier,C215493,NCIT_C215493,Administrable Product Identifier,"A sequence of characters used to identify, nam...",Concrete,1
5,AdministrableProductProperty,AdministrableProductProperty,C215494,NCIT_C215494,Administrable Product Property,A characteristic from a set of characteristics...,Concrete,0
6,Administration,Administration,C25409,NCIT_C25409,Administration,"The act of dispensing, applying, or tendering ...",Concrete,0
7,AliasCode,AliasCode,C201344,NCIT_C201344,Alias Code,An alternative symbol or combination of symbol...,Concrete,0
8,AnalysisPopulation,AnalysisPopulation,C188814,NCIT_C188814,Analysis Population,A target study population on which an analysis...,Concrete,0
9,AssignedPerson,AssignedPerson,C215496,NCIT_C215496,Assigned Person,An individual person who is allotted or appoin...,Concrete,0


## Step 2 — attribute dictionary

Each property's domain, range, NCIt anchors (EVS + OBO PURL), structural metadata
(`relationshipType`, `modelName`, `modelRepresentation`), cardinality
(reconstructed from `owl:Restriction` triples), and codelist
binding columns. Single-class ranges only — polymorphic union ranges
land in step 3.


In [3]:
ATTR_SPARQL = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?prop ?domain ?label ?range ?ncit_evs ?ncit_obo ?prefLabel ?definition
       ?relationshipType ?modelName ?modelRepresentation
       ?boundCodelist_evs ?boundCodelist_obo ?boundCodelistNote
WHERE {
  { ?prop a owl:DatatypeProperty } UNION { ?prop a owl:ObjectProperty }
  ?prop rdfs:domain ?domain ; rdfs:label ?label .
  OPTIONAL { ?prop rdfs:range ?range . FILTER(isIRI(?range)) }
  OPTIONAL {
    ?prop skos:exactMatch ?ncit_evs .
    FILTER(STRSTARTS(STR(?ncit_evs), "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"))
  }
  OPTIONAL {
    ?prop skos:exactMatch ?ncit_obo .
    FILTER(STRSTARTS(STR(?ncit_obo), "http://purl.obolibrary.org/obo/NCIT_"))
  }
  OPTIONAL { ?prop skos:prefLabel ?prefLabel }
  OPTIONAL { ?prop skos:definition ?definition }
  OPTIONAL { ?prop usdm:relationshipType ?relationshipType }
  OPTIONAL { ?prop usdm:modelName ?modelName }
  OPTIONAL { ?prop usdm:modelRepresentation ?modelRepresentation }
  OPTIONAL {
    ?prop usdm:boundCodelist ?boundCodelist_evs .
    FILTER(STRSTARTS(STR(?boundCodelist_evs), "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"))
  }
  OPTIONAL {
    ?prop usdm:boundCodelist ?boundCodelist_obo .
    FILTER(STRSTARTS(STR(?boundCodelist_obo), "http://purl.obolibrary.org/obo/NCIT_"))
  }
  OPTIONAL { ?prop usdm:boundCodelistNote ?boundCodelistNote }
}
"""

CARD_SPARQL = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?prop ?card ?minCard ?maxCard
WHERE {
  ?domain rdfs:subClassOf ?r .
  ?r a owl:Restriction ; owl:onProperty ?prop .
  OPTIONAL { ?r owl:cardinality ?card }
  OPTIONAL { ?r owl:minCardinality ?minCard }
  OPTIONAL { ?r owl:maxCardinality ?maxCard }
}
"""

def cardinality_string(card, min_card, max_card):
    if card is not None:
        return str(card)
    lo = str(min_card) if min_card is not None else "0"
    hi = str(max_card) if max_card is not None else "*"
    return f"{lo}..{hi}"

card_lookup = {}
for r in g.query(CARD_SPARQL):
    card_lookup[str(r["prop"])] = cardinality_string(r["card"], r["minCard"], r["maxCard"])

rows = []
for r in g.query(ATTR_SPARQL):
    rows.append({
        "prop":                short(r["prop"]),
        "domain":              short(r["domain"]),
        "label":               str(r["label"]),
        "range":               short(r["range"]),
        "cardinality":         card_lookup.get(str(r["prop"]), "0..*"),
        "ncit_evs":            short(r["ncit_evs"]),
        "ncit_obo":            short(r["ncit_obo"]),
        "prefLabel":           str(r["prefLabel"]) if r["prefLabel"] else None,
        "definition":          str(r["definition"]) if r["definition"] else None,
        "relationshipType":    str(r["relationshipType"]) if r["relationshipType"] else None,
        "modelName":           str(r["modelName"]) if r["modelName"] else None,
        "modelRepresentation": str(r["modelRepresentation"]) if r["modelRepresentation"] else None,
        "boundCodelist_evs":   short(r["boundCodelist_evs"]),
        "boundCodelist_obo":   short(r["boundCodelist_obo"]),
        "boundCodelistNote":   str(r["boundCodelistNote"]) if r["boundCodelistNote"] else None,
    })

attrs_df = pd.DataFrame(rows)
print(f"{len(attrs_df)} property rows")
attrs_df.head(10)


693 property rows


,prop,domain,label,range,cardinality,ncit_evs,ncit_obo,prefLabel,definition,relationshipType,modelName,modelRepresentation,boundCodelist_evs,boundCodelist_obo,boundCodelistNote
0,Abbreviation-id,Abbreviation,Abbreviation-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None
1,Abbreviation-abbreviatedText,Abbreviation,Abbreviation-abbreviatedText,string,1,C215487,NCIT_C215487,Text Abbreviation Value,The literal value of an instance of unstructur...,Value,abbreviatedText,Attribute,None,None,None
2,Abbreviation-expandedText,Abbreviation,Abbreviation-expandedText,string,1,C215569,NCIT_C215569,Abbreviation Long Name,The full literal representation of the abbrevi...,Value,expandedText,Attribute,None,None,None
3,Abbreviation-instanceType,Abbreviation,Abbreviation-instanceType,string,1,None,None,None,None,Value,None,None,None,None,None
4,Activity-id,Activity,Activity-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None
5,Activity-name,Activity,Activity-name,string,1,C188842,NCIT_C188842,Study Activity Name,"The literal identifier (i.e., distinctive desi...",Value,name,Attribute,None,None,None
6,Activity-label,Activity,Activity-label,string,0..1,C207458,NCIT_C207458,Study Activity Label,The short descriptive designation for the stud...,Value,label,Attribute,None,None,None
7,Activity-description,Activity,Activity-description,string,0..1,C70960,NCIT_C70960,Study Activity Description,A narrative representation of the study activity.,Value,description,Attribute,None,None,None
8,Activity-instanceType,Activity,Activity-instanceType,string,1,None,None,None,None,Value,None,None,None,None,None
9,Address-id,Address,Address-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None


## Step 3 — polymorphic associations

Four properties in DDF-RA v4.0.0 have multi-class `Type` lists, rendered
as `owl:unionOf`. The SPARQL above filtered them out (their range is a
blank node, not an IRI). Here we resolve the union members and update
the `range` column to `Class1 | Class2 | ...`.


In [4]:
UNION_SPARQL = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?prop ?member
WHERE {
  ?prop rdfs:range ?range .
  ?range owl:unionOf ?list .
  ?list rdf:rest*/rdf:first ?member .
}
"""

union_members = {}
for r in g.query(UNION_SPARQL):
    union_members.setdefault(short(r["prop"]), []).append(short(r["member"]))

for prop, members in union_members.items():
    attrs_df.loc[attrs_df["prop"] == prop, "range"] = " | ".join(members)

print(f"{len(union_members)} polymorphic associations:")
for prop in union_members:
    row = attrs_df[attrs_df["prop"] == prop].iloc[0]
    attr_short = row["prop"].split("-", 1)[-1]
    print(f"  {row['domain']}.{attr_short:24s} -> {row['range']}")


4 polymorphic associations:
  Condition.contextIds               -> Activity | ScheduledActivityInstance
  Condition.appliesToIds             -> BiomedicalConceptCategory | Procedure | Activity | BiomedicalConcept | BiomedicalConceptSurrogate
  ProductOrganizationRole.appliesToIds             -> AdministrableProduct | MedicalDevice
  StudyRole.appliesToIds             -> StudyVersion | StudyDesign


## Step 4 — flat data dictionary

Merge `classes_df` and `attrs_df` on the class IRI (the property's
domain). Result: one row per (class, attribute) with full metadata.
Exported as `data_dictionary.csv` in the `examples/` directory.
Tracked in git — regenerated on every run, so diffs reflect source changes.


In [5]:
dict_df = attrs_df.merge(
    classes_df[["class", "ncit_evs", "ncit_obo", "prefLabel", "modifier"]].rename(
        columns={
            "class":     "domain",
            "ncit_evs":  "class_ncit_evs",
            "ncit_obo":  "class_ncit_obo",
            "prefLabel": "class_prefLabel",
            "modifier":  "class_modifier",
        }
    ),
    on="domain",
    how="left",
)

column_order = [
    "domain", "class_modifier", "class_ncit_evs", "class_ncit_obo", "class_prefLabel",
    "prop", "label", "range", "cardinality",
    "ncit_evs", "ncit_obo", "prefLabel", "definition",
    "relationshipType", "modelName", "modelRepresentation",
    "boundCodelist_evs", "boundCodelist_obo", "boundCodelistNote",
]
dict_df = dict_df[column_order]

CSV_PATH = "data_dictionary.csv"
dict_df.to_csv(CSV_PATH, index=False)

print(f"flat dictionary: {len(dict_df)} rows × {len(dict_df.columns)} columns")
print(f"columns: {list(dict_df.columns)}")
print(f"wrote {CSV_PATH}")
dict_df.head(10)


flat dictionary: 693 rows × 19 columns
columns: ['domain', 'class_modifier', 'class_ncit_evs', 'class_ncit_obo', 'class_prefLabel', 'prop', 'label', 'range', 'cardinality', 'ncit_evs', 'ncit_obo', 'prefLabel', 'definition', 'relationshipType', 'modelName', 'modelRepresentation', 'boundCodelist_evs', 'boundCodelist_obo', 'boundCodelistNote']
wrote data_dictionary.csv


,domain,class_modifier,class_ncit_evs,class_ncit_obo,class_prefLabel,prop,label,range,cardinality,ncit_evs,ncit_obo,prefLabel,definition,relationshipType,modelName,modelRepresentation,boundCodelist_evs,boundCodelist_obo,boundCodelistNote
0,Abbreviation,Concrete,C42610,NCIT_C42610,Abbreviation,Abbreviation-id,Abbreviation-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None
1,Abbreviation,Concrete,C42610,NCIT_C42610,Abbreviation,Abbreviation-abbreviatedText,Abbreviation-abbreviatedText,string,1,C215487,NCIT_C215487,Text Abbreviation Value,The literal value of an instance of unstructur...,Value,abbreviatedText,Attribute,None,None,None
2,Abbreviation,Concrete,C42610,NCIT_C42610,Abbreviation,Abbreviation-expandedText,Abbreviation-expandedText,string,1,C215569,NCIT_C215569,Abbreviation Long Name,The full literal representation of the abbrevi...,Value,expandedText,Attribute,None,None,None
3,Abbreviation,Concrete,C42610,NCIT_C42610,Abbreviation,Abbreviation-instanceType,Abbreviation-instanceType,string,1,None,None,None,None,Value,None,None,None,None,None
4,Activity,Concrete,C71473,NCIT_C71473,Study Activity,Activity-id,Activity-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None
5,Activity,Concrete,C71473,NCIT_C71473,Study Activity,Activity-name,Activity-name,string,1,C188842,NCIT_C188842,Study Activity Name,"The literal identifier (i.e., distinctive desi...",Value,name,Attribute,None,None,None
6,Activity,Concrete,C71473,NCIT_C71473,Study Activity,Activity-label,Activity-label,string,0..1,C207458,NCIT_C207458,Study Activity Label,The short descriptive designation for the stud...,Value,label,Attribute,None,None,None
7,Activity,Concrete,C71473,NCIT_C71473,Study Activity,Activity-description,Activity-description,string,0..1,C70960,NCIT_C70960,Study Activity Description,A narrative representation of the study activity.,Value,description,Attribute,None,None,None
8,Activity,Concrete,C71473,NCIT_C71473,Study Activity,Activity-instanceType,Activity-instanceType,string,1,None,None,None,None,Value,None,None,None,None,None
9,Address,Concrete,C25407,NCIT_C25407,Address,Address-id,Address-id,string,1,None,None,None,None,Value,id,Attribute,None,None,None


## Step 5 — sample explorations

Three concrete questions answered by filtering `dict_df`:

1. What attributes does `Study` have, with their codelist bindings?
2. Which properties have a definition but no NCIt anchor?
3. Which properties have both an NCIt anchor and a structured codelist binding?


In [6]:
# Q1: All Study attributes with bindings
# (_evs / _obo columns are always paired — validation enforces D4 — so
#  filtering on the _evs form alone is safe; it carries the bare C-code.)
study = dict_df[dict_df["domain"] == "Study"][
    ["prop", "range", "cardinality", "ncit_evs", "boundCodelist_evs", "boundCodelistNote"]
]
print(f"Study has {len(study)} declared attributes")
study


Study has 8 declared attributes


,prop,range,cardinality,ncit_evs,boundCodelist_evs,boundCodelistNote
240,Study-id,string,1,None,None,None
241,Study-name,string,1,C68631,None,None
242,Study-label,string,0..1,C207479,None,None
243,Study-description,string,0..1,C142704,None,None
244,Study-instanceType,string,1,None,None,None
555,Study-versions,StudyVersion,0..*,None,None,None
556,Study-documentedBy,StudyDefinitionDocument,0..*,None,None,None
557,Study-extensionAttributes,ExtensionAttribute,0..*,None,None,None


In [7]:
# Q2: Properties with Definition but no NCI C-Code
no_ncit = dict_df[
    dict_df["definition"].notna() & dict_df["ncit_evs"].isna()
][["domain", "prop", "definition"]]
print(f"{len(no_ncit)} properties have a definition but no NCI C-Code")
no_ncit.head(10)


126 properties have a definition but no NCI C-Code


,domain,prop,definition
362,Activity,Activity-definedProcedures,A USDM relationship between the Activity and P...
363,Activity,Activity-biomedicalConceptIds,A USDM relationship between the Activity and B...
364,Activity,Activity-nextId,A USDM relationship within the Activity class ...
365,Activity,Activity-timelineId,A USDM relationship between the Activity and S...
366,Activity,Activity-childIds,A USDM relationship within the Activity class ...
367,Activity,Activity-previousId,A USDM relationship within the Activity class ...
368,Activity,Activity-bcSurrogateIds,A USDM relationship between the Activity and B...
369,Activity,Activity-bcCategoryIds,A USDM relationship between the Activity and B...
378,AdministrableProduct,AdministrableProduct-identifiers,A USDM relationship between the AdministrableP...
379,AdministrableProduct,AdministrableProduct-properties,A USDM relationship between the AdministrableP...


In [8]:
# Q3: NCI-anchored AND structured codelist binding
both = dict_df[
    dict_df["ncit_evs"].notna() & dict_df["boundCodelist_evs"].notna()
][["domain", "prop", "ncit_evs", "ncit_obo", "boundCodelist_evs", "boundCodelist_obo", "boundCodelistNote"]]
print(f"{len(both)} properties have both an attribute-level NCI anchor and a structured codelist binding")
both


45 properties have both an attribute-level NCI anchor and a structured codelist binding


,domain,prop,ncit_evs,ncit_obo,boundCodelist_evs,boundCodelist_obo,boundCodelistNote
373,AdministrableProduct,AdministrableProduct-administrableDoseForm,C215576,NCIT_C215576,C66726,NCIT_C66726,Y (SDTM Terminology Codelist C66726)
374,AdministrableProduct,AdministrableProduct-sourcing,C215578,NCIT_C215578,C215483,NCIT_C215483,Y (C215483)
375,AdministrableProduct,AdministrableProduct-productDesignation,C215579,NCIT_C215579,C207418,NCIT_C207418,Y (C207418)
383,AdministrableProductProperty,AdministrableProductProperty-type,C215585,NCIT_C215585,C215479,NCIT_C215479,Y (C215479)
387,Administration,Administration-frequency,C89081,NCIT_C89081,C71113,NCIT_C71113,Y (SDTM Terminology Codelist C71113)
388,Administration,Administration-route,C38114,NCIT_C38114,C66729,NCIT_C66729,Y (SDTM Terminology Codelist C66729)
432,EligibilityCriterion,EligibilityCriterion-category,C83016,NCIT_C83016,C66797,NCIT_C66797,Y (SDTM Terminology Codelist C66797)
439,Encounter,Encounter-type,C188839,NCIT_C188839,C188728,NCIT_C188728,Y (C188728)
440,Encounter,Encounter-environmentalSettings,C188840,NCIT_C188840,C127262,NCIT_C127262,Y (SDTM Terminology Codelist C127262)
441,Encounter,Encounter-contactModes,C188841,NCIT_C188841,C171445,NCIT_C171445,Y (SDTM Terminology Codelist C171445)


## Discussion

What this shows: the model is queryable as data, not just readable as
documentation. With v0.1, codelist binding columns appear in the
dictionary alongside the structural metadata — a CT-bound property is
as easy to find as a property with a particular cardinality.
Since v0.4.0 (decision D4) every NCIt reference carries two IRI forms —
the EVS identifier and the resolvable OBO PURL — and the dictionary
keeps them as paired `_evs` / `_obo` columns, one row per attribute.

Deliberately *not* shown:

- **Inheritance walking.** A class's *effective* attribute set (its own
  attributes plus those inherited via `rdfs:subClassOf`) requires
  walking the class hierarchy. Case 3 (coverage / gap analysis) takes
  that on.
- **Codelist permitted-term resolution.** `usdm:boundCodelist` gives
  the codelist's NCIt C-Code; the actual permitted Code values live in
  CDISC Library RDF or the Library REST API. A future Library/CT
  extension closes that loop.

Pointers: case 2 deepens the codelist binding view (forward and reverse
lookup, source-category breakdown); case 3 takes on inheritance and
the unbound-Code-typed-property gap analysis.
